# Economy-wide impact assessment

Prepare Italy's 2022 supply-use system for the twin
[CVXLab tutorial](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html).
This notebook covers the MARIO data preparation and separate import accounts.
Continue in CVXLab to define electricity-generation scenarios and solve the optimisation model.

The resulting system has 189 products, 163 activities, 9 final-demand categories,
12 factors of production and one GHG indicator. Electricity commodities are combined
into one product. All production activities, including electricity technologies, are retained.

## Environment and source files

Use Python 3.12 and the verified MARIO development revision below. Its separate-import
configuration is required by this notebook. Create an environment, then install and launch Jupyter.

```sh
python -m pip install "mariopy @ git+https://github.com/it-is-me-mario/MARIO.git@f137208feb031d8eb452a27f33336424767d17b5" jupyterlab
jupyter lab
```

Download `MRSUT_2022.zip` and `IOT_2022_ixi.zip` from the official
[EXIOBASE 3.8.2 record](https://zenodo.org/records/5589597) and put both archives in
`source_data` beside this notebook. Keep them zipped. The IOT archive supplies the matching
satellite accounts. The combined downloads occupy approximately 1 GB.

EXIOBASE 3.8.2 was published in 2021. Its 2022 tables are estimated extensions of the
historical series. The source data are licensed under CC-BY-SA-4.0. Retain the
[EXIOBASE attribution](https://doi.org/10.1111/jiec.12715) when reusing derived results.

In [ ]:
from pathlib import Path
import inspect

import mario
import numpy as np
import pandas as pd

assert "trade_resolution" in inspect.signature(mario.Database.to_single_region).parameters, (
    "Install the MARIO development revision specified above and restart the kernel."
)

source_dir = Path("source_data")
output_dir = Path("results")
sut_path = source_dir / "MRSUT_2022.zip"
iot_path = source_dir / "IOT_2022_ixi.zip"
assert sut_path.is_file() and iot_path.is_file(), "Place both EXIOBASE archives in source_data."

## Parse the monetary supply-use table

The SUT archive supplies economic flows. Satellite accounts come from the industry-by-industry
IOT of the same release and year. MARIO attaches them to the corresponding activities.

In [ ]:
db = mario.parse_exiobase(
    path=str(sut_path),
    table="SUT",
    unit="Monetary",
    add_extensions=str(iot_path),
)

## Extract Italy with separate imports

`import_representation="separate"` creates native import matrices alongside the factors
of production. `imports_by="commodity"` retains the imported-product classification.
`trade_resolution="aggregate"` combines foreign origins and export destinations.

`IMP_a` records imports used by activities, `IMP_c` imports entering domestic commodity
supply and `IMP_Y` imports delivered directly to final demand. `Va` retains the
domestic factors of production. Imported inputs are excluded from `U` and `Yc`.

In [ ]:
db.to_single_region(
    "IT",
    imports_by="commodity",
    import_representation="separate",
    trade_resolution="aggregate",
)

## Characterise greenhouse gases

Combine the EXIOBASE greenhouse-gas accounts with the built-in IPCC AR6 GWP100 profile.
The retained row measures kg CO2-equivalent. Direct final-demand emissions remain in `EY`.

In [ ]:
ghg_label = "GHG AR6 GWP-100"
db.calc_ghg(profile="exiobase_monetary", label=ghg_label)
db.keep_extensions([ghg_label])

## Combine electricity products

Map the twelve EXIOBASE electricity commodities to `Electricity`. Keep the native activity
classification so the CVXLab exercise can vary individual generation technologies.
The same commodity mapping is applied to import accounts. `zero_output_epsilon=None`
sums the observed flows directly.

In [ ]:
commodities = db.get_index("Commodity")
electricity = db.search("Commodity", "electricity ")
assert len(electricity) == 12

commodity_mapping = pd.DataFrame({"Aggregation": commodities}, index=commodities)
commodity_mapping.loc[electricity, "Aggregation"] = "Electricity"
db.aggregate(
    io={"Commodity": commodity_mapping},
    levels=["Commodity"],
    calc_all=False,
    zero_output_epsilon=None,
)

## Check the dimensions and import coefficients

These checks use the full classification described in the CVXLab tutorial. MARIO computes
`imp_a` by dividing each column of `IMP_a` by the corresponding activity output `Xa`.

In [ ]:
expected_sizes = {
    "Region": 1,
    "Commodity": 189,
    "Activity": 163,
    "Consumption category": 9,
    "Factor of production": 12,
    "Satellite account": 1,
}
sizes = {name: len(db.get_index(name)) for name in expected_sizes}
assert sizes == expected_sizes, sizes
assert "imports" not in db.get_index("Factor of production")

imports = db.query("IMP_a")
coefficients = db.query("imp_a")
activity_output = db.query("Xa").iloc[:, 0]
assert imports.columns.equals(activity_output.index)
assert imports.index.equals(coefficients.index)
np.testing.assert_allclose(
    coefficients.to_numpy() * activity_output.to_numpy(),
    imports.to_numpy(),
    rtol=1e-10,
    atol=1e-6,
)
pd.Series(sizes, name="Members").to_frame()

## Export the matrices and matching sets

Each CSV contains one MARIO matrix in long format, with labelled coordinates, `Value`
and `Unit` columns. The workbook contains the transformed sets in their current order.
`S` has activities on rows and commodities on columns. `U` uses commodities on rows
and activities on columns. `Xa` and `Xc` provide their output vectors.

MARIO exports monetary flows in EUR and GHG flows in kg CO2-equivalent. The CVXLab
tutorial presents M EUR and kt CO2-equivalent. Dividing each of these flow values by
1,000,000 gives that presentation scale. Preserve consistent units throughout the model.

In [ ]:
matrices = [
    "S", "U", "Yc", "Va", "Ea", "VY", "EY", "Xa", "Xc",
    "IMP_a", "IMP_c", "IMP_Y",
]
db.export(
    matrices=matrices,
    scenarios="baseline",
    format="csv",
    split="matrix",
    path=str(output_dir),
    include_meta=False,
)
sets_path = db.export_sets(path=output_dir / "sets.xlsx")
assert sets_path.is_file()

exported = {}
for path in sorted(output_dir.glob("*.csv")):
    frame = pd.read_csv(path)
    names = frame["Matrix"].unique().tolist()
    if len(names) == 1 and names[0] in matrices:
        assert np.isfinite(frame["Value"]).all()
        exported[names[0]] = {"File": path.name, "Rows": len(frame), "Unit": ", ".join(frame["Unit"].unique())}
assert set(exported) == set(matrices), exported
pd.DataFrame.from_dict(exported, orient="index")

## Continue in CVXLab

Use these matrices and `sets.xlsx` for the reference system in
[Economy-wide impact assessment](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html).
The CVXLab model settings must use the same full classification, separate import accounts
and units. The notebook's exports retain the MARIO matrix names shown above.

Consult MARIO's
[single-region API](https://www.marioproject.com/en/latest/api_document/mario.Database.to_single_region.html),
[export API](https://www.marioproject.com/en/latest/api_document/mario.Database.export.html) and the
[development export guide](https://github.com/it-is-me-mario/MARIO/blob/f137208feb031d8eb452a27f33336424767d17b5/doc/source/user_guide/inspection/export_and_roundtrip.rst)
for the configuration and output formats used here.